# SmartDelivery360 — Spark Transformation Notebook

**Fabric Lakehouse notebook** implementing the bronze → silver → gold medallion pipeline
for the SmartDelivery360 order & delivery-delay analysis.

> Run this inside a Fabric notebook attached to a Lakehouse. Update the `LAKEHOUSE_PATH` / table
> names in the first code cell to match your workspace before running.

## 0. Setup

Point this at the file once you've uploaded `data/sample_orders.csv` to **Files** in your Lakehouse
(e.g. `Files/raw/sample_orders.csv`). In Fabric, `spark` is already available — no need to create a session.

In [ ]:
# Path to the raw CSV inside your Lakehouse Files section
# e.g. "Files/raw/sample_orders.csv" once uploaded via the Fabric UI
RAW_CSV_PATH = "Files/raw/sample_orders.csv"

BRONZE_TABLE = "bronze_orders"
SILVER_TABLE = "silver_orders"

## 1. Bronze Layer — raw ingestion

Land the raw CSV as-is into a Delta table. No cleaning yet — bronze should mirror the source.

In [ ]:
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_CSV_PATH)
)

df_raw.write.format("delta").mode("overwrite").saveAsTable(BRONZE_TABLE)

print(f"Bronze row count: {df_raw.count()}")
df_raw.printSchema()

## 2. Silver Layer — clean, dedupe, transform (Spark SQL)

- Cast/standardize types
- Drop exact duplicate `OrderID`s (keep first occurrence)
- Normalize `DelayedOrder` to a boolean `IsDelayed`
- Derive `DelayRateFlag` and trim string columns

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW vw_silver_orders AS
WITH deduped AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY OrderID ORDER BY OrderDate) AS rn
    FROM {BRONZE_TABLE}
)
SELECT
    OrderID,
    TRIM(CustomerID)                       AS CustomerID,
    CAST(OrderDate AS DATE)                AS OrderDate,
    TRIM(Region)                           AS Region,
    TRIM(OrderChannel)                     AS OrderChannel,
    TRIM(ProductCategory)                  AS ProductCategory,
    CAST(Quantity AS INT)                  AS Quantity,
    ROUND(CAST(TotalRevenue AS DOUBLE), 2)            AS TotalRevenue,
    ROUND(CAST(RevenueFromDiscounted AS DOUBLE), 2)   AS RevenueFromDiscounted,
    ROUND(CAST(TotalShipment AS DOUBLE), 2)           AS TotalShipment,
    CASE WHEN TRIM(IsPremiumCustomer) = 'Yes' THEN true ELSE false END AS IsPremiumCustomer,
    CASE WHEN TRIM(DelayedOrder) = 'Yes' THEN true ELSE false END       AS IsDelayed,
    CAST(DelayDays AS INT)                 AS DelayDays
FROM deduped
WHERE rn = 1
""")

df_silver = spark.table("vw_silver_orders")
df_silver.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

print(f"Silver row count: {df_silver.count()}")
display(df_silver.limit(10))

## 3. Gold Layer — business-ready aggregates for Power BI

Three gold tables, one per dashboard pivot: region, channel, and premium-customer performance.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE gold_region_summary AS
SELECT
    Region,
    COUNT(*)                                   AS TotalOrders,
    SUM(CASE WHEN IsDelayed THEN 1 ELSE 0 END)  AS DelayedOrders,
    ROUND(SUM(TotalRevenue), 2)                 AS TotalRevenue,
    ROUND(SUM(RevenueFromDiscounted), 2)        AS RevenueFromDiscounted,
    ROUND(SUM(TotalShipment), 2)                AS TotalShipment
FROM {SILVER_TABLE}
GROUP BY Region
ORDER BY TotalRevenue DESC
""")

display(spark.table("gold_region_summary"))

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE gold_channel_summary AS
SELECT
    OrderChannel,
    COUNT(*)                                   AS TotalOrders,
    SUM(CASE WHEN IsDelayed THEN 1 ELSE 0 END)  AS DelayedOrders,
    ROUND(SUM(TotalRevenue), 2)                 AS TotalRevenue,
    ROUND(
        100.0 * SUM(CASE WHEN IsDelayed THEN 1 ELSE 0 END) / COUNT(*), 1
    )                                            AS DelayRatePct
FROM {SILVER_TABLE}
GROUP BY OrderChannel
ORDER BY TotalRevenue DESC
""")

display(spark.table("gold_channel_summary"))

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE gold_premium_summary AS
SELECT
    IsPremiumCustomer,
    COUNT(DISTINCT CustomerID)          AS Customers,
    COUNT(*)                            AS TotalOrders,
    ROUND(SUM(TotalRevenue), 2)         AS TotalRevenue
FROM {SILVER_TABLE}
GROUP BY IsPremiumCustomer
""")

display(spark.table("gold_premium_summary"))

## 4. Sanity checks against the published dashboard

These should land close to the KPI cards on the Power BI report (17 customers, 25 orders,
121 units, 11 delayed orders). Small variance is expected — this notebook runs against the
anonymized sample dataset, not the original source data.

In [ ]:
from pyspark.sql import functions as F

summary = df_silver.agg(
    F.countDistinct("CustomerID").alias("Customers"),
    F.count("*").alias("TotalOrders"),
    F.sum("Quantity").alias("TotalQuantity"),
    F.round(F.sum("TotalShipment"), 2).alias("TotalShipment"),
    F.sum(F.when(F.col("IsDelayed"), 1).otherwise(0)).alias("DelayedOrders"),
)
display(summary)

## Next steps

- Point Power BI at `gold_region_summary`, `gold_channel_summary`, and `gold_premium_summary`
  via Direct Lake mode for the fastest refresh.
- Add a `gold_delay_rate_by_region` table (delay **rate** rather than raw count) — flagged as a
  known gap in the current dashboard, since raw counts understate risk in smaller regions.
- Wire this notebook into a Fabric Data Pipeline for scheduled runs.